# Module 1 --- Python Fundamentals & PyTorch Tensors

**Course:** Python and Machine Learning with PyTorch
**Companion slides:** `Presentation_1_Python_Basics.pdf`

---

### What you will do here

1. **Setup** --- install libraries, import them, pick a device.
2. **Theory in practice** --- Python syntax drills, then tensor creation, tensor
   operations, broadcasting, NumPy interop, and CPU vs GPU timing.
3. **Challenge** --- write your own normalisation function using only tensor operations.

### How to use this notebook

Run every cell in order with `Shift + Enter`. Cells marked **TODO** are for you to
complete. For the GPU section, enable a GPU first:
`Runtime > Change runtime type > Hardware accelerator > GPU`.

## 1. Setup

In [ ]:
# Colab already ships these, so the install is usually instant.
# Uncomment if you run this notebook somewhere else.
# !pip install torch torchvision matplotlib pandas scikit-learn
!pip install torch torchvision matplotlib pandas scikit-learn --quiet

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print("torch version :", torch.__version__)
print("numpy version :", np.__version__)

# Reproducibility: run this before anything random.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Device configuration. Every later cell reuses this variable.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU name     :", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Runtime > Change runtime type > Hardware accelerator > GPU")

## 2. Theory in Practice

### 2.1 Python syntax refresher

Variables, the four container types, and f-strings.

In [ ]:
# --- Scalars -----------------------------------------------------------
n_samples     = 150          # int
learning_rate = 0.01         # float
model_name    = "resnet18"   # str
is_training   = True         # bool

print(f"{model_name} | lr = {learning_rate:.3f} | n = {n_samples}")
print("true division :", n_samples / 4)    # 37.5
print("floor division:", n_samples // 4)   # 37   <- use this for indices

# --- List: ordered, mutable -------------------------------------------
losses = [2.31, 1.87, 1.45, 1.02]
losses.append(0.88)
print("\nlist        :", losses)
print("first / last:", losses[0], losses[-1])
print("slice [1:3] :", losses[1:3])        # end index is EXCLUDED
print("min / max   :", min(losses), max(losses))

# --- Tuple: ordered, immutable (this is how shapes are written) --------
shape = (3, 224, 224)
print("\ntuple:", shape, "-> channels =", shape[0])

# --- Dict: key -> value (this is how configs are written) --------------
config = {"lr": 0.01, "epochs": 10, "batch_size": 32}
config["optimizer"] = "adam"
print("\ndict:", config)
print("get with default:", config.get("momentum", 0.9))

# --- Set: unordered, unique -------------------------------------------
print("\nset:", {0, 1, 2, 2, 1})

In [ ]:
# --- Control flow ------------------------------------------------------
accuracy = 0.83

if accuracy > 0.90:
    verdict = "Excellent"
elif accuracy > 0.75:
    verdict = "Good enough for a baseline"
else:
    verdict = "Needs more training"
print(verdict)

# for + enumerate + zip
names = ["resnet", "vgg", "mlp"]
scores = [0.91, 0.87, 0.72]

for i, (name, score) in enumerate(zip(names, scores)):
    print(f"  {i}: {name:8s} -> {score:.2f}")

# while
loss, step = 10.0, 0
while loss > 1.0 and step < 100:
    loss *= 0.8
    step += 1
print(f"\nreached loss={loss:.3f} after {step} steps")

# List comprehension: the Pythonic loop
squared = [s ** 2 for s in scores]
good    = [n for n, s in zip(names, scores) if s > 0.8]
print("squared:", [round(x, 3) for x in squared])
print("good   :", good)

In [ ]:
# --- Functions, *args and **kwargs -------------------------------------

def mse(pred, target):
    """Mean squared error between two lists of numbers."""
    return sum((p - t) ** 2 for p, t in zip(pred, target)) / len(pred)


def total(*args):
    """*args collects any number of POSITIONAL arguments into a tuple."""
    print("  args received:", args)
    return sum(args)


def build_model(**kwargs):
    """**kwargs collects any number of KEYWORD arguments into a dict."""
    print("  kwargs received:", kwargs)
    return kwargs.get("depth", 1)


print("mse   :", mse([1.0, 2.0, 3.0], [1.1, 1.9, 3.2]))
print("total :", total(1, 2, 3, 4))
print("depth :", build_model(depth=4, width=128))

### 2.2 Creating tensors

A tensor is an n-dimensional array. Its three attributes worth checking whenever
something breaks are `.shape`, `.dtype` and `.device`.

In [ ]:
torch.manual_seed(SEED)

a = torch.tensor([[1., 2.], [3., 4.]])   # from a Python list
z = torch.zeros(3, 4)
o = torch.ones(2, 5)
e = torch.eye(3)                         # identity
r = torch.rand(2, 3)                     # uniform in [0, 1)
n = torch.randn(2, 3)                    # standard normal
s = torch.arange(0, 10, 2)               # 0, 2, 4, 6, 8
l = torch.linspace(0, 1, steps=5)

print("a =\n", a)
print("\narange   :", s)
print("linspace :", l)

print("\n--- the three attributes to check ---")
print("shape :", a.shape)     # torch.Size([2, 2])
print("dtype :", a.dtype)     # torch.float32
print("device:", a.device)    # cpu
print("ndim  :", a.ndim)      # 2
print("numel :", a.numel())   # 4

In [ ]:
# Rank 0 to 4: read every shape from the RIGHT.
examples = {
    "scalar (a loss)          ": torch.tensor(0.53),
    "vector (class scores)    ": torch.randn(10),
    "matrix (batch of vectors)": torch.randn(32, 784),
    "one RGB image            ": torch.randn(3, 224, 224),
    "a batch of images        ": torch.randn(32, 3, 224, 224),
}
for name, t in examples.items():
    print(f"{name} rank={t.ndim}  shape={tuple(t.shape)}")

### 2.3 Exercise --- build a 3x3 matrix and multiply

This is the exercise announced on the slides. Note the difference between the
**elementwise** product `*`, the **matrix** product `@`, and the **dot** product
of two vectors.

In [ ]:
# A 3x3 matrix built explicitly, and a second one from a range.
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.],
                  [7., 8., 10.]])
B = torch.arange(1, 10, dtype=torch.float32).reshape(3, 3)

print("A =\n", A)
print("\nB =\n", B)

print("\nA + B (elementwise) =\n", A + B)
print("\nA * B (elementwise, NOT matrix product) =\n", A * B)
print("\nA @ B (matrix product) =\n", A @ B)

# Confirm that @ really is the textbook definition, for entry (0, 0):
manual_00 = sum(A[0, k] * B[k, 0] for k in range(3))
print("\nmanual (A@B)[0,0] =", manual_00.item(), " torch says:", (A @ B)[0, 0].item())

In [ ]:
# Dot product of two VECTORS
u = torch.tensor([1., 2., 3.])
v = torch.tensor([4., 5., 6.])

print("torch.dot(u, v) :", torch.dot(u, v).item())     # 32.0
print("(u * v).sum()   :", (u * v).sum().item())       # same thing
print("u @ v           :", (u @ v).item())             # same thing again

# Matrix-vector product
print("\nA @ u =", A @ u)

In [ ]:
# Reductions, transpose and reshaping
print("sum        :", A.sum().item())
print("mean       :", A.mean().item())
print("std        :", A.std().item())
print("column sums:", A.sum(dim=0))    # collapses dim 0 -> one value per column
print("row sums   :", A.sum(dim=1))    # collapses dim 1 -> one value per row
print("argmax     :", A.argmax().item(), "(index in the FLATTENED tensor)")

print("\nA.T =\n", A.T)
print("\nflattened   :", A.reshape(-1))          # -1 means "infer this dimension"
print("as 1x9      :", A.view(1, 9).shape)
print("with batch  :", A.unsqueeze(0).shape)     # (1, 3, 3)
print("squeezed    :", A.unsqueeze(0).squeeze(0).shape)

### 2.4 Broadcasting

Shapes are compared **from the right**. Two dimensions are compatible if they are
equal, or if one of them is 1. The size-1 dimension is then repeated without
copying memory.

In [ ]:
M = torch.arange(12, dtype=torch.float32).reshape(3, 4)
row = torch.tensor([10., 20., 30., 40.])    # shape (4,)   -> broadcast over rows
col = torch.tensor([[1.], [2.], [3.]])      # shape (3, 1) -> broadcast over columns

print("M =\n", M)
print("\nM + row  (shape (3,4) + (4,)) =\n", M + row)
print("\nM + col  (shape (3,4) + (3,1)) =\n", M + col)

# The realistic use case: subtract a per-column mean
print("\ncolumn means:", M.mean(dim=0))
print("centred M =\n", M - M.mean(dim=0))

# A shape that does NOT broadcast
try:
    M + torch.tensor([1., 2., 3.])      # (3,4) vs (3,) -> 4 against 3, incompatible
except RuntimeError as err:
    print("\nExpected error:", err)

### 2.5 NumPy and PyTorch

`torch.from_numpy` and `.numpy()` **share memory**. `torch.tensor(...)` **copies**.
Getting this wrong causes bugs that are very hard to trace.

In [ ]:
arr = np.array([1., 2., 3.])

shared = torch.from_numpy(arr)   # shares memory with arr
copied = torch.tensor(arr)       # independent copy

arr[0] = 999.0                   # modify the NumPy array in place

print("numpy array    :", arr)
print("shared tensor  :", shared, "  <- changed too")
print("copied tensor  :", copied, "  <- unaffected")

# Going back to NumPy (needed for matplotlib)
t = torch.randn(3)
print("\ntensor -> numpy:", t.numpy())
print("dtype note: numpy defaults to float64, torch to float32")
print("arr dtype:", arr.dtype, "| torch.from_numpy dtype:", shared.dtype)

### 2.6 CPU versus GPU

The same matrix multiplication, timed on both devices. Change `SIZE` below and
re-run to see how the gap widens with problem size.

In [ ]:
# ---- user-adjustable parameter -------------------------------------------
SIZE = 2000       # try 500, 2000, 4000
# --------------------------------------------------------------------------

x_cpu = torch.randn(SIZE, SIZE)
y_cpu = torch.randn(SIZE, SIZE)

start = time.time()
_ = x_cpu @ y_cpu
cpu_time = time.time() - start
print(f"CPU: {cpu_time:.4f} s")

if device.type == "cuda":
    x_gpu = x_cpu.to(device)
    y_gpu = y_cpu.to(device)

    _ = x_gpu @ y_gpu                 # warm-up: the first CUDA call is slow
    torch.cuda.synchronize()          # CUDA is asynchronous, so we must wait

    start = time.time()
    _ = x_gpu @ y_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start

    print(f"GPU: {gpu_time:.4f} s")
    print(f"speed-up: {cpu_time / gpu_time:.1f}x")
else:
    print("No GPU available, skipping the comparison.")

In [ ]:
# Devices must match. This is one of the most common PyTorch errors.
if device.type == "cuda":
    on_gpu = torch.randn(2, 2, device=device)
    on_cpu = torch.randn(2, 2)
    try:
        _ = on_gpu + on_cpu
    except RuntimeError as err:
        print("Expected error:", str(err)[:120], "...")
    print("\nFixed with .to(device):", (on_gpu + on_cpu.to(device)).shape)
else:
    print("GPU not enabled, nothing to demonstrate here.")

### 2.7 A first look at autograd

Preview of Module 2. PyTorch records operations on tensors flagged with
`requires_grad=True`, then differentiates by walking the record backwards.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1        # y = (x + 1)^2,  so dy/dx = 2x + 2

y.backward()
print("y      =", y.item())
print("dy/dx  =", x.grad.item(), " (analytically 2*3 + 2 = 8)")

# Visualise the function and the tangent at x = 3
xs = torch.linspace(-5, 5, 200)
ys = xs ** 2 + 2 * xs + 1
slope, x0, y0 = 8.0, 3.0, 16.0

plt.figure(figsize=(5, 3.2))
plt.plot(xs.numpy(), ys.numpy(), label="y = x^2 + 2x + 1")
plt.plot(xs.numpy(), (y0 + slope * (xs - x0)).numpy(), "--", label="tangent at x=3")
plt.scatter([x0], [y0], color="red", zorder=5)
plt.ylim(-5, 50); plt.legend(); plt.grid(alpha=0.3)
plt.title("autograd recovers the slope")
plt.tight_layout(); plt.show()

## 3. Challenge

Three tasks, increasing in difficulty. Solutions follow, but try them first.

**Challenge 1.** Write `standardise(X)` that takes a 2-D tensor of shape `(n, d)`
and returns a tensor where every **column** has mean 0 and standard deviation 1.
Use only tensor operations, no Python loop.

**Challenge 2.** Write `pairwise_sq_dist(A, B)` returning a matrix `D` where
`D[i, j]` is the **squared** Euclidean distance between `A[i]` and `B[j]`.
Use broadcasting and `unsqueeze`. You will reuse this idea in Module 4.

**Challenge 3.** Write `min_max_scale(X)` mapping every column into `[0, 1]`,
and handle the case of a constant column without producing `NaN`.

In [ ]:
# ---- user-adjustable parameters ------------------------------------------
N_SAMPLES = 8
N_FEATURES = 3
# --------------------------------------------------------------------------

torch.manual_seed(SEED)
X = torch.randn(N_SAMPLES, N_FEATURES) * torch.tensor([1.0, 10.0, 100.0]) + 5.0
print("X shape:", X.shape)
print("column means:", X.mean(dim=0))
print("column stds :", X.std(dim=0))

In [ ]:
# TODO Challenge 1
def standardise(X):
    """Return X with every column centred and scaled to unit variance."""
    # your code here
    raise NotImplementedError


# TODO Challenge 2
def pairwise_sq_dist(A, B):
    """A of shape (m, d), B of shape (n, d) -> D of shape (m, n)."""
    # your code here
    raise NotImplementedError


# TODO Challenge 3
def min_max_scale(X):
    """Map every column of X into [0, 1]."""
    # your code here
    raise NotImplementedError

### Solutions

Only look after you have tried.

In [ ]:
def standardise(X):
    mu = X.mean(dim=0)                 # shape (d,)
    sigma = X.std(dim=0)               # shape (d,)
    return (X - mu) / (sigma + 1e-8)   # epsilon guards against a zero std


def pairwise_sq_dist(A, B):
    # (m, 1, d) - (1, n, d) -> (m, n, d), then sum over the feature axis
    diff = A.unsqueeze(1) - B.unsqueeze(0)
    return diff.pow(2).sum(dim=2)


def min_max_scale(X):
    lo = X.min(dim=0).values
    hi = X.max(dim=0).values
    span = (hi - lo).clamp(min=1e-8)   # a constant column gives span 0
    return (X - lo) / span


# --- checks ---------------------------------------------------------------
Xs = standardise(X)
print("after standardise, means:", Xs.mean(dim=0).round(decimals=5))
print("after standardise, stds :", Xs.std(dim=0).round(decimals=5))

A = torch.randn(4, 3)
B = torch.randn(6, 3)
D = pairwise_sq_dist(A, B)
print("\npairwise_sq_dist shape:", D.shape)
print("matches torch.cdist?  :", torch.allclose(D.sqrt(), torch.cdist(A, B), atol=1e-4))

Xm = min_max_scale(X)
print("\nafter min_max_scale, mins:", Xm.min(dim=0).values)
print("after min_max_scale, maxs:", Xm.max(dim=0).values)

const_col = torch.ones(5, 1)
print("constant column stays finite:", torch.isfinite(min_max_scale(const_col)).all().item())

In [ ]:
# Visual proof that standardisation matters: three features on wildly
# different scales, before and after.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

for j in range(N_FEATURES):
    axes[0].scatter([j] * N_SAMPLES, X[:, j].numpy(), s=25)
    axes[1].scatter([j] * N_SAMPLES, Xs[:, j].numpy(), s=25)

axes[0].set_title("raw features"); axes[1].set_title("standardised features")
for ax in axes:
    ax.set_xlabel("feature index"); ax.set_xticks(range(N_FEATURES)); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

### Optional: interactive input

`input()` blocks until you type something and press Enter. Handy for a quick
demo, but remember it stops an automated run, so keep it out of training scripts.

In [ ]:
raw = input("Enter a tensor size (integer, e.g. 4): ")

try:
    k = int(raw)
    T = torch.randn(k, k)
    print(f"\nrandom {k}x{k} tensor:\n{T}")
    print(f"\ntrace = {T.trace().item():.4f}")
    print(f"T @ T.T is symmetric: {torch.allclose(T @ T.T, (T @ T.T).T, atol=1e-6)}")
except ValueError:
    print("That was not an integer.")

## Recap

| Concept | Key syntax |
|---|---|
| Check a tensor | `.shape`, `.dtype`, `.device` |
| Elementwise vs matrix product | `A * B` vs `A @ B` |
| Reduce along an axis | `A.sum(dim=0)`, `A.mean(dim=1)` |
| Reshape | `.reshape()`, `.view()`, `.unsqueeze()`, `.squeeze()` |
| Move to GPU | `x.to(device)` |
| NumPy interop | `torch.from_numpy` shares, `torch.tensor` copies |
| Gradients | `requires_grad=True`, then `.backward()` |

**Next:** Module 2, where a Python class becomes a neural network.